# Notebook 02: Experiment 1 latency analysis scaffold

This notebook is the canonical analysis entry point for **Experiment 1**: the
**Cell A / B / C MCP-overhead comparison**.

Right now it serves two jobs:

1. verify that the expected benchmark artifact layout exists for Cells A / B / C
2. export a stable readiness snapshot under `results/metrics/`

Once real benchmark captures land, the same notebook should roll forward into the
actual latency comparison without needing a structural rewrite.


In [ ]:
import json
import platform
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


In [ ]:
def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "benchmarks").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from notebook path")

REPO_ROOT = find_repo_root(Path.cwd())
BENCHMARKS_DIR = REPO_ROOT / "benchmarks"
RESULTS_METRICS_DIR = REPO_ROOT / "results" / "metrics"
RESULTS_FIGURES_DIR = REPO_ROOT / "results" / "figures"
RESULTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"matplotlib: {matplotlib.__version__}")


## Expected Experiment 1 cells

Notebook 02 is intentionally scoped to the three Experiment 1 conditions:

- **Cell A**: direct-tool baseline
- **Cell B**: MCP baseline
- **Cell C**: optimized MCP path

It does not read Cell Y or Cell Z. Those belong to Experiment 2 and Notebook 03.


In [ ]:
CELL_SPECS = [
    {"cell": "A", "label": "Direct", "path": BENCHMARKS_DIR / "cell_A_direct"},
    {"cell": "B", "label": "MCP baseline", "path": BENCHMARKS_DIR / "cell_B_mcp_baseline"},
    {"cell": "C", "label": "MCP optimized", "path": BENCHMARKS_DIR / "cell_C_mcp_optimized"},
]

def count_jsonl_rows(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open() as handle:
        return sum(1 for line in handle if line.strip())

def load_json_if_present(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text())

def scan_cell(cell_spec: dict) -> dict:
    cell_dir = cell_spec["path"]
    raw_dir = cell_dir / "raw"
    run_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()]) if raw_dir.exists() else []
    latest_run = run_dirs[-1] if run_dirs else None
    summary = load_json_if_present(cell_dir / "summary.json")
    config = load_json_if_present(cell_dir / "config.json")
    latencies_path = latest_run / "latencies.jsonl" if latest_run else None
    return {
        "cell": cell_spec["cell"],
        "label": cell_spec["label"],
        "cell_dir": str(cell_dir.relative_to(REPO_ROOT)),
        "config_present": (cell_dir / "config.json").exists(),
        "summary_present": (cell_dir / "summary.json").exists(),
        "raw_run_count": len(run_dirs),
        "latest_run_id": latest_run.name if latest_run else "",
        "latency_rows": count_jsonl_rows(latencies_path) if latencies_path else 0,
        "run_status": summary.get("run_status") if summary else None,
        "scenario_set_name": summary.get("scenario_set_name") if summary else None,
        "model_id": summary.get("model_id") if summary else None,
        "wandb_run_url": summary.get("wandb_run_url") if summary else None,
        "summary_latency_mean": summary.get("latency_seconds_mean") if summary else None,
        "summary_latency_p95": summary.get("latency_seconds_p95") if summary else None,
        "config_experiment_family": config.get("experiment_family") if config else None,
    }

availability_df = pd.DataFrame(scan_cell(spec) for spec in CELL_SPECS)
availability_df


In [ ]:
availability_path = RESULTS_METRICS_DIR / "notebook02_cell_availability.csv"
availability_df.to_csv(availability_path, index=False)
print(f"Wrote {availability_path.relative_to(REPO_ROOT)}")
display(availability_df)


## Readiness gate

This notebook should only claim Experiment 1 findings once **all three** cells have:

- a committed `config.json`
- a committed `summary.json`
- at least one raw run directory with `latencies.jsonl` rows

Until then, the notebook stays in preflight mode and only exports the availability snapshot.


In [ ]:
availability_df["analysis_ready"] = (
    availability_df["config_present"]
    & availability_df["summary_present"]
    & (availability_df["raw_run_count"] > 0)
    & (availability_df["latency_rows"] > 0)
)

ready_cells = availability_df.loc[availability_df["analysis_ready"], "cell"].tolist()
missing_cells = availability_df.loc[~availability_df["analysis_ready"], "cell"].tolist()

print(f"Ready cells: {ready_cells if ready_cells else 'none'}")
print(f"Missing cells: {missing_cells if missing_cells else 'none'}")

analysis_ready = len(missing_cells) == 0
if not analysis_ready:
    print("Notebook 02 remains in scaffold mode until Cells A, B, and C all have committed benchmark artifacts.")


In [ ]:
def load_cell_latencies(cell_spec: dict) -> pd.DataFrame:
    cell_dir = cell_spec["path"]
    raw_dir = cell_dir / "raw"
    run_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()])
    latest_run = run_dirs[-1]
    rows = []
    with (latest_run / "latencies.jsonl").open() as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            row["cell"] = cell_spec["cell"]
            row["label"] = cell_spec["label"]
            row["run_id"] = latest_run.name
            rows.append(row)
    return pd.DataFrame(rows)

if analysis_ready:
    latency_df = pd.concat([load_cell_latencies(spec) for spec in CELL_SPECS], ignore_index=True)
    latency_summary = (
        latency_df.groupby(["cell", "label"], as_index=False)["latency_seconds"]
        .agg(["count", "mean", "median", "max"])
        .reset_index()
    )
    latency_summary.columns = ["cell", "label", "count", "latency_mean", "latency_median", "latency_max"]
    summary_path = RESULTS_METRICS_DIR / "notebook02_latency_summary.csv"
    latency_summary.to_csv(summary_path, index=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(latency_summary["label"], latency_summary["latency_mean"], color=["#4c78a8", "#f58518", "#54a24b"])
    ax.set_title("Experiment 1 mean latency by cell")
    ax.set_ylabel("Latency (seconds)")
    ax.set_xlabel("Condition")
    fig.tight_layout()

    figure_path = RESULTS_FIGURES_DIR / "notebook02_latency_comparison.png"
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    print(f"Wrote {summary_path.relative_to(REPO_ROOT)}")
    print(f"Wrote {figure_path.relative_to(REPO_ROOT)}")
    display(latency_summary)
else:
    print("Skipping latency aggregation because at least one Experiment 1 cell still lacks committed artifacts.")


## What changes later

When `#25` and the first real Experiment 1 captures land, this notebook should not need a new
shape. The next update should mostly be about richer metrics and plots:

- split end-to-end latency from tool-only latency if both are available
- join in profiling-linked metadata from WandB and `summary.json`
- export the final paper-facing latency figure(s) under `results/figures/`
- keep `results/metrics/notebook02_cell_availability.csv` as the cheap preflight artifact
